# Eviny Charging Curve Classification LSTM 
#### by Sebastian Einar Salas Røkholt
----

### Index
**01 - Setup** </br>
**02 - Data Exploration, Wrangling and Preprocessing**</br>
**03 - Modelling**</br>
**04 - Model Evaluation and Selection**</br>
**+++**</br>

## 01 - Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42


## 02 - Data Exploration and Wrangling

In [2]:
df = pd.read_parquet("winter2023/data/etron55.parquet")
print(df.shape)
print(df.head())

(722665, 7)
                  timestamp   soc  power  energy  location_coordinates_lat  \
0  2020-01-11T12:37:21.000Z  40.0  89.44    0.32                 59.668261   
1  2020-01-11T12:38:21.000Z  41.0  92.75    1.84                 59.668261   
2  2020-01-11T12:39:21.000Z  43.0  94.81    3.41                 59.668261   
3  2020-01-11T12:40:21.000Z  45.0  95.68    5.00                 59.668261   
4  2020-01-11T12:41:21.000Z  47.0  96.88    6.60                 59.668261   

   location_coordinates_long  id  
0                   9.652725   1  
1                   9.652725   1  
2                   9.652725   1  
3                   9.652725   1  
4                   9.652725   1  


In [3]:
def preprocess(
    data,
    trim_ends=True,
    interpolate=False,
    drop_zero=True,
):
    """
    Take a dataframe of charging curves, each curve identified by the column
    charging_id_anon. Do various filling, interpolation etc. on the frame.
    Note: the power column is linearly interpolated, the rest is just ffilled

    Parameters
    ----------
    data : TYPE
        DESCRIPTION.
    trim_ends : TYPE, optional
        DESCRIPTION. The default is True.
    interpolate : TYPE, optional
        DESCRIPTION. The default is True.
    drop_zero : TYPE, optional
        DESCRIPTION. The default is True.

    Returns
    -------
    data_with_all : TYPE
        DESCRIPTION.

    """
    data.soc = pd.Categorical(data.soc)
    data = data.drop_duplicates()

    data_with_all_soc = data.groupby(["id", "soc"]).sum(
        numeric_only=True
    )

    islands_with_data = data_with_all_soc
    if trim_ends:
        # Now we have soc 1-100 for all sessions. Remove leading and trailing zeros for each session
        islands_with_data = (
            (data_with_all_soc["power"] != 0)
            .groupby(level=0)
            .apply(lambda x: np.trim_zeros(x))
        ).reset_index(level=1)

    islands_with_data = islands_with_data.reset_index(level=1)[
        "soc"
    ].to_frame()

    data_with_all = data.merge(
        islands_with_data, on=["id", "soc"], how="right"
    )
    if interpolate:
        data_with_all["power"] = data_with_all["power"].interpolate()
        data_with_all = data_with_all.interpolate(
            "ffill"
        )  # Safe since the first (and last) row for each session is not null

    if drop_zero:
        data_with_all = data_with_all[data_with_all["power"] != 0]

    return data_with_all

data = preprocess(df)
data.head()

C:\Users\sebas\AppData\Local\Temp\ipykernel_21660\2239447660.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data_with_all_soc = data.groupby(["id", "soc"]).sum(


,timestamp,soc,power,energy,location_coordinates_lat,location_coordinates_long,id
0,2020-01-11T12:37:21.000Z,40.0,89.44,0.32,59.668261,9.652725,1
1,2020-01-11T12:38:21.000Z,41.0,92.75,1.84,59.668261,9.652725,1
2,NaN,42.0,NaN,NaN,NaN,NaN,1
3,2020-01-11T12:39:21.000Z,43.0,94.81,3.41,59.668261,9.652725,1
4,NaN,44.0,NaN,NaN,NaN,NaN,1


In [4]:
# Reordering the columns
cols = data.columns.tolist()
cols_reordered = cols[-1:] + cols[:-1]
# Remove lat and long as I don't want temperatures in the model yet
# Remove energy because it is simply an aggregate of power over time (kWh), so it doesn't improve predictive ability
cols = [col for col in cols_reordered if col not in ["location_coordinates_lat", "location_coordinates_long", "energy"]]
df = data[cols]
df.head()

,id,timestamp,soc,power
0,1,2020-01-11T12:37:21.000Z,40.0,89.44
1,1,2020-01-11T12:38:21.000Z,41.0,92.75
2,1,NaN,42.0,NaN
3,1,2020-01-11T12:39:21.000Z,43.0,94.81
4,1,NaN,44.0,NaN


In [5]:
# Explore the data
print("Shape: ", df.shape)
print("Number of charging sessions: ", len(pd.unique(df["id"])))
print("Time interval: ", df["timestamp"].min(), " to ", df["timestamp"].max())
display(df.info())

Shape:  (1232748, 4)
Number of charging sessions:  29474


TypeError: '<=' not supported between instances of 'str' and 'float'

In [ ]:
# Set appropriate data types
print(df.head())
pd.options.mode.chained_assignment = None  # Temporarily suppress SettingWithCopyWarning
df["soc"] = df["soc"].astype("float64")
df["power"] = df["power"].astype("float64")
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["id"] = df["id"].astype("category")

# Convert "timestamp" to "minutes_elapsed" for each session
df.sort_values(by=['id', 'timestamp'], inplace=True)  # Ensure that the df is sorted
df['minutes_elapsed'] = df.groupby('id')['timestamp'].transform(lambda x: (x - x.min()).dt.total_seconds() / 60)
df.drop("timestamp", axis=1)

df.head()
pd.options.mode.chained_assignment = "warn"
df.dtypes


   id                 timestamp   soc   power
0   1  2020-01-11T12:37:21.000Z  40.0  89.440
1   1  2020-01-11T12:38:21.000Z  41.0  92.750
2   1  2020-01-11T12:38:21.000Z  42.0  93.780
3   1  2020-01-11T12:39:21.000Z  43.0  94.810
4   1  2020-01-11T12:39:21.000Z  44.0  95.245


C:\Users\sebas\AppData\Local\Temp\ipykernel_23876\3106603410.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['minutes_elapsed'] = df.groupby('id')['timestamp'].transform(lambda x: (x - x.min()).dt.total_seconds() / 60)


id                            category
timestamp          datetime64[ns, UTC]
soc                            float64
power                          float64
minutes_elapsed                float64
dtype: object

In [ ]:
df.head(10)

,id,timestamp,soc,power,minutes_elapsed
0,1,2020-01-11 12:37:21+00:00,40.0,89.440,0.0
1,1,2020-01-11 12:38:21+00:00,41.0,92.750,1.0
2,1,2020-01-11 12:38:21+00:00,42.0,93.780,1.0
3,1,2020-01-11 12:39:21+00:00,43.0,94.810,2.0
4,1,2020-01-11 12:39:21+00:00,44.0,95.245,2.0
5,1,2020-01-11 12:40:21+00:00,45.0,95.680,3.0
6,1,2020-01-11 12:40:21+00:00,46.0,96.280,3.0
7,1,2020-01-11 12:41:21+00:00,47.0,96.880,4.0
8,1,2020-01-11 12:42:21+00:00,48.0,97.620,5.0
9,1,2020-01-11 12:42:21+00:00,49.0,98.245,5.0


#### Preparing the data for modelling

In [ ]:
# Splitting the data into training, validation and test sets
def split_data(df, test_size=0.2, validation_size=0.1):
    train_val_df, test_df = train_test_split(df, test_size=test_size, shuffle=False, stratify=None)
    validation_size = validation_size / (1 - test_size)  # Adjust validation size based on the remaining dataset after test split
    train_df, val_df = train_test_split(train_val_df, test_size=validation_size, shuffle=False)
    return train_df, val_df, test_df

train_df, val_df, test_df = split_data(df)

In [ ]:
# Performs data normalisation
features = ['power', 'soc', 'minutes_elapsed']
scaler = MinMaxScaler(feature_range=(0, 1))
train_df[features] = scaler.fit_transform(train_df[features])  # Only fits the scaler on training data (Prevents info leakage)
val_df[features] = scaler.transform(val_df[features])
test_df[features] = scaler.transform(test_df[features])

In [ ]:
# For creating sequences (charging sessions) to feed into the LSTM
def create_sequences(df, sequence_length=5):
    all_xs = []
    all_ys = []
    for session_id, session_df in df.groupby('id', observed=False):
        session_features = session_df[['power', 'soc', 'minutes_elapsed']].values
        session_targets = session_df[['power', 'soc']].values

        xs, ys = [], []
        for i in range(len(session_features) - sequence_length):
            x = session_features[i:(i + sequence_length)]
            y = session_targets[i + sequence_length]
            xs.append(x)
            ys.append(y)

        all_xs.extend(xs)
        all_ys.extend(ys)
    
    return np.array(all_xs), np.array(all_ys)

# Create sequences for each set
sequence_length = 15  # Lookback window for predictions will be 15 time steps/minutes
X_train, y_train = create_sequences(train_df, sequence_length)
X_val, y_val = create_sequences(val_df, sequence_length)
X_test, y_test = create_sequences(test_df, sequence_length)

C:\Users\sebas\AppData\Local\Temp\ipykernel_23876\1650400335.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for session_id, session_df in df.groupby('id'):
C:\Users\sebas\AppData\Local\Temp\ipykernel_23876\1650400335.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for session_id, session_df in df.groupby('id'):
C:\Users\sebas\AppData\Local\Temp\ipykernel_23876\1650400335.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this 

In [ ]:
print("First 5 training sequences: \n(Columns are 'power', 'soc', 'minutes elapsed') \n", X_train[:5])
print("\n\nTrue output for first 5 training sequences:\n", y_train[:5],)

First 5 training sequences: 
(Columns are 'power', 'soc', 'minutes elapsed') 
 [[[0.33394331 0.39393939 0.        ]
  [0.34630466 0.4040404  0.00625   ]
  [0.35015125 0.41414141 0.00625   ]
  [0.35399783 0.42424242 0.0125    ]
  [0.35562236 0.43434343 0.0125    ]
  [0.35724689 0.44444444 0.01875   ]
  [0.35948762 0.45454545 0.01875   ]
  [0.36172835 0.46464646 0.025     ]
  [0.36449191 0.47474747 0.03125   ]
  [0.36682601 0.48484848 0.03125   ]
  [0.3691601  0.49494949 0.0375    ]
  [0.37048586 0.50505051 0.0375    ]
  [0.37181163 0.51515152 0.04375   ]
  [0.37207305 0.52525253 0.04375   ]
  [0.37233447 0.53535354 0.05      ]
  [0.37222243 0.54545455 0.05625   ]
  [0.3720357  0.55555556 0.05625   ]
  [0.37184897 0.56565657 0.0625    ]
  [0.37166225 0.57575758 0.0625    ]
  [0.37147552 0.58585859 0.06875   ]
  [0.37356687 0.5959596  0.06875   ]
  [0.37565821 0.60606061 0.075     ]
  [0.37588229 0.61616162 0.08125   ]
  [0.37586361 0.62626263 0.08125   ]
  [0.37584494 0.63636364 0.0875  

In [ ]:
batch_size = 128
num_workers = 16

# Converts from Numpy arrays to PyTorch tensors
train_features = torch.Tensor(X_train)
train_targets = torch.Tensor(y_train)
val_features = torch.Tensor(X_val)
val_targets = torch.Tensor(y_val)

# Creates TensorDatasets
train_dataset = TensorDataset(train_features, train_targets)
val_dataset = TensorDataset(val_features, val_targets)

# Creates DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

## Modelling

In [ ]:
# Defines the general model architecture and the forward pass
class MultivariateLSTM(nn.Module):
    def __init__(self, input_size, hidden_layer_size, output_size, num_layers):
        super(MultivariateLSTM, self).__init__()
        self.hidden_layer_size = hidden_layer_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_layer_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_layer_size, output_size)

    # Forward propagation
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_layer_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_layer_size).to(x.device)
        
        # LSTM layer output
        out, _ = self.lstm(x, (h0, c0))
        
        # Last time step hidden state
        out = self.linear(out[:, -1, :])
        return out


In [ ]:
# Defines the training loop (gradient descent + backpropagation)
def train_model(model, train_loader, val_loader, num_epochs=100, learning_rate=0.0001, plot_loss=True):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model.train()
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    if plot_loss == True:
        # For plotting
        train_losses = list()
        val_losses = list()
    
    # Runs the training loop
    for epoch in range(num_epochs):  # For every pass through the entire dataset
        epoch_train_loss, epoch_val_loss = 0.0, 0.0
        
        for inputs, labels in train_loader:  # For every batch
            inputs, labels = inputs.to(device), labels.to(device)  # Moves data to the training device
            optimizer.zero_grad()  # (Re)sets gradients to 0 so that they don't compound
            outputs = model(inputs)  # Forward pass / Runs predictions
            train_loss = criterion(outputs, labels)  # Calculate the batch loss
            epoch_train_loss += train_loss.item()  # Add loss to this epoch's total training loss
            train_loss.backward()  # Calculates the gradient of the loss function wrt params with backpropagation
            optimizer.step()  # Updates the model parameters based on the gradient, step size and optimization strategy
        
        # For printing the validation loss during training
        model.eval()  # Set model to evaluation mode
        with torch.no_grad():  # No learning here
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                val_loss = criterion(outputs, labels)
                epoch_val_loss += val_loss.item()
        
        # Averages the loss per epoch
        avg_epoch_val_loss = epoch_val_loss / len(val_loader)
        avg_epoch_train_loss = epoch_train_loss / len(train_loader)

        # Print the epoch's average loss every 5 epochs
        if (epoch + 1) % 5 == 0: 
            print(f'Epoch {epoch+1}, Training loss: {avg_epoch_train_loss}, Validation Loss: {avg_epoch_val_loss}')
        if plot_loss == True and epoch > 5:  # I don't want to plot the first few epochs because loss is very high
            # Store loss for plotting purposes
            train_losses.append(avg_epoch_train_loss)
            val_losses.append(avg_epoch_val_loss)

        model.train()  # Set model back to train mode before loading next batch
    
    print('Training complete.')

    if plot_loss == True: 
        # Create a DataFrame from the loss lists
        data = {
            'Epoch': range(1, len(train_losses) + 1),
            'Training Loss': train_losses,
            'Validation Loss': val_losses
        }
        df_losses = pd.DataFrame(data).melt(id_vars=["Epoch"], var_name="Type", value_name="Loss")

        # Plot using Seaborn
        plt.figure(figsize=(10, 6))
        sns.lineplot(data=df_losses, x='Epoch', y='Loss', hue='Type')
        plt.title('Training and Validation Loss Over Number of Training Epochs')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend(title='Loss type')
        plt.show()


In [ ]:
# Setting the model hyperparameters
input_size = 3  # power, soc, minutes_elapsed
hidden_layer_size = 60
output_size = 2  # Predicting power and soc for the next time step
num_layers = 4  # Number of LSTM layers

# Defining the network architecture and hyperparameters
model = MultivariateLSTM(input_size, hidden_layer_size, output_size, num_layers)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

MultivariateLSTM(
  (lstm): LSTM(3, 60, num_layers=4, batch_first=True)
  (linear): Linear(in_features=60, out_features=2, bias=True)
)

In [ ]:
# Running the training loop
model_path = '.\Models\LSTM_model_2.pth'
# print(f"Training on {device}")
# train_model(model, train_loader, val_loader, num_epochs=150, plot_loss=True)
# torch.save(model.state_dict(), model_path)

## Prediction

In [ ]:
# Find the longest sequence in the test data
test_session_ids = test_df["id"].unique()
print(test_session_ids)

# Filter the test DataFrame for the selected session
max_id = 0
max_len = 0
for i in range(len(test_session_ids)):
    selected_sessions_id = test_session_ids[i]
    selected_sessions_df = test_df[test_df['id'] == selected_sessions_id]
    if len(selected_sessions_df) > max_len:
        max_len = len(selected_sessions_df)
        max_id = selected_sessions_id

print(f"Max length: {max_len}, ID: {max_id}")

[23790, 23791, 23792, 23793, 23794, ..., 29838, 29839, 29840, 29841, 29842]
Length: 5983
Categories (29474, int64): [1, 2, 3, 4, ..., 29839, 29840, 29841, 29842]
Max length: 312, ID: 26586


In [ ]:
# Selecting a random charging sessions
# test_session_ids = test_df["id"].unique()

# Filter the test DataFrame for the selected sessions
# selected_sessions_id = [test_session_ids[max_id]]
max_id = 29839
selected_sessions_df = test_df[test_df['id'] == max_id]
print(len(selected_sessions_df))

# Performs normalization
# selected_sessions_df[features] = scaler.transform(selected_sessions_df[features])
# Create sequences 
X_test_selected, y_test_selected = create_sequences(selected_sessions_df, sequence_length)
print(f"Number of X sequences: {len(X_test_selected)}", f"\nNumber of y_true: {len(y_test_selected)}")
selected_sessions_df.head()

70


C:\Users\sebas\AppData\Local\Temp\ipykernel_23876\1650400335.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for session_id, session_df in df.groupby('id'):


Number of X sequences: 40 
Number of y_true: 40


,id,timestamp,soc,power,minutes_elapsed
1236485,29839,2023-06-18 17:54:52+00:00,0.101010,0.514994,0.00000
1236486,29839,2023-06-18 17:54:52+00:00,0.111111,0.517272,0.00000
1236487,29839,2023-06-18 17:54:52+00:00,0.121212,0.519550,0.00000
1236488,29839,2023-06-18 17:55:52+00:00,0.131313,0.521828,0.00625
1236489,29839,2023-06-18 17:55:52+00:00,0.141414,0.522787,0.00625


In [ ]:
print(X_test_selected[:5])
print("\n\n", X_train[:5])

[[[0.13253912 0.26262626 0.        ]
  [0.13253912 0.26262626 0.00208333]
  [0.13253912 0.26262626 0.00416667]
  [0.13436905 0.26262626 0.00625   ]
  [0.13287523 0.26262626 0.00833333]
  [0.13470516 0.27272727 0.01041667]
  [0.13436905 0.27272727 0.0125    ]
  [0.13504127 0.27272727 0.01458333]
  [0.13321134 0.27272727 0.01666667]
  [0.13470516 0.28282828 0.01875   ]
  [0.13470516 0.28282828 0.02083333]
  [0.13321134 0.28282828 0.02291667]
  [0.13321134 0.28282828 0.025     ]
  [0.13470516 0.29292929 0.02708333]
  [0.13470516 0.29292929 0.02916667]
  [0.13321134 0.29292929 0.03125   ]
  [0.13504127 0.29292929 0.03333333]
  [0.13321134 0.3030303  0.03541667]
  [0.13470516 0.3030303  0.0375    ]
  [0.13321134 0.3030303  0.03958333]
  [0.13354745 0.3030303  0.04166667]
  [0.13321134 0.31313131 0.04375   ]
  [0.13321134 0.31313131 0.04583333]
  [0.13321134 0.31313131 0.04791667]
  [0.13470516 0.31313131 0.05      ]
  [0.13321134 0.32323232 0.05208333]
  [0.13504127 0.32323232 0.05416667]
 

In [ ]:
# Load the model
model.load_state_dict(torch.load(model_path))
model.eval()

MultivariateLSTM(
  (lstm): LSTM(3, 60, num_layers=4, batch_first=True)
  (linear): Linear(in_features=60, out_features=2, bias=True)
)

In [ ]:
# Run predictions
session_predictions = list()
with torch.no_grad():
    for sequence in X_test_selected:
        # print("Input: ", sequence)
        seq_tensor = torch.tensor(sequence, dtype=torch.float).unsqueeze(0).to(device)
        prediction = model(seq_tensor).cpu().numpy()
        # print("Prediction: ", prediction)
        session_predictions.append(prediction)

print(session_predictions)

[array([[0.5387991, 0.405115 ]], dtype=float32), array([[0.53895324, 0.41514954]], dtype=float32), array([[0.5401246 , 0.42517117]], dtype=float32), array([[0.541105  , 0.43517366]], dtype=float32), array([[0.54087335, 0.44520497]], dtype=float32), array([[0.54176235, 0.45526907]], dtype=float32), array([[0.5424481 , 0.46526626]], dtype=float32), array([[0.54252523, 0.47537604]], dtype=float32), array([[0.54341966, 0.48554024]], dtype=float32), array([[0.54432076, 0.49556848]], dtype=float32), array([[0.5492237, 0.5055868]], dtype=float32), array([[0.5474084, 0.5156778]], dtype=float32), array([[0.54875046, 0.52585775]], dtype=float32), array([[0.54933923, 0.5361294 ]], dtype=float32), array([[0.5493832, 0.5460319]], dtype=float32), array([[0.5527441, 0.5561529]], dtype=float32), array([[0.55439264, 0.56620973]], dtype=float32), array([[0.55439156, 0.5762736 ]], dtype=float32), array([[0.55596954, 0.58647615]], dtype=float32), array([[0.5566171, 0.5966148]], dtype=float32), array([[0.5

In [ ]:
pred_df = pd.DataFrame([pred[0] for pred in session_predictions], columns=["Power", "soc"])
pred_df

,Power,soc
0,0.538799,0.405115
1,0.538953,0.415150
2,0.540125,0.425171
3,0.541105,0.435174
4,0.540873,0.445205
5,0.541762,0.455269
6,0.542448,0.465266
7,0.542525,0.475376
8,0.543420,0.485540
9,0.544321,0.495568


In [ ]:
# Reverse normalisation of predictions
pred_df["minutes_elapsed"] = selected_sessions_df.tail(len(pred_df)).minutes_elapsed.values
unscaled_pred_df = pd.DataFrame(scaler.inverse_transform(pred_df), columns=features)
display(unscaled_pred_df.head())
# Reverse normalisation of actuals
unscaled_true_df = pd.DataFrame(scaler.inverse_transform(selected_sessions_df[features]), columns=features)
display(unscaled_true_df.head())


,power,soc,minutes_elapsed
0,144.294237,41.106386,10.0
1,144.335510,42.099804,10.0
2,144.649163,43.091945,11.0
3,144.911678,44.082192,11.0
4,144.849657,45.075292,11.0


,power,soc,minutes_elapsed
0,137.920000,11.0,0.0
1,138.530000,12.0,0.0
2,139.140000,13.0,0.0
3,139.750000,14.0,1.0
4,140.006667,15.0,1.0


In [ ]:
display(unscaled_true_df.head(40))
display(unscaled_pred_df.head(10))


,power,soc,minutes_elapsed
0,137.920000,11.0,0.0
1,138.530000,12.0,0.0
2,139.140000,13.0,0.0
3,139.750000,14.0,1.0
4,140.006667,15.0,1.0
5,140.263333,16.0,1.0
6,140.520000,17.0,2.0
7,140.756667,18.0,2.0
8,140.993333,19.0,2.0
9,141.230000,20.0,3.0


,power,soc,minutes_elapsed
0,144.294237,41.106386,10.0
1,144.335510,42.099804,10.0
2,144.649163,43.091945,11.0
3,144.911678,44.082192,11.0
4,144.849657,45.075292,11.0
5,145.087705,46.071638,12.0
6,145.271329,47.061359,12.0
7,145.291981,48.062228,12.0
8,145.531482,49.068484,13.0
9,145.772770,50.061280,14.0


In [ ]:
# Create a figure and a set of subplots
fig, ax1 = plt.subplots()

# Plotting Power
ax1.set_xlabel('Minutes elapsed in current charging session')
ax1.set_ylabel('power')
ax1.plot(unscaled_pred_df['minutes_elapsed'], unscaled_pred_df['power'], color="darkorange", label='Predicted Power')
ax1.plot(unscaled_true_df['minutes_elapsed'], unscaled_true_df['power'], color='darkred', label='True Power', alpha=0.5)
ax1.tick_params(axis='y')


# Plotting SOC on dual axis
ax2 = ax1.twinx()
ax2.plot(unscaled_pred_df['minutes_elapsed'], unscaled_pred_df['soc'], color="darkblue", label='Predicted SOC')
ax2.plot(unscaled_true_df['minutes_elapsed'], unscaled_true_df['soc'], color='darkblue', label='True SOC', alpha=0.5)

lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc='upper left')

plt.title(f'Predictions vs. Actuals for session {max_id}')
plt.xlabel('Time Step')
plt.ylabel('Power / SOC')
# plt.legend()
plt.show()


## Anomaly Detection

In [ ]:
def detect_anomalies(model, new_data, threshold):
    model.eval()  # Ensure the model is in evaluation mode
    anomalies = []
    
    # Assuming new_data is preprocessed and in the correct format
    with torch.no_grad():
        for sequence in new_data:
            # Make prediction
            sequence = torch.tensor(sequence, dtype=torch.float).unsqueeze(0).to(device)  # Add batch dimension
            prediction = model(sequence)
            
            # Calculate error
            actual = sequence[:, -1, :2].squeeze().cpu().numpy()  # Last timestep, actual power and soc
            prediction = prediction.squeeze().cpu().numpy()
            error = np.mean((actual - prediction)**2)
            
            # Check if error exceeds threshold
            if error > threshold:
                anomalies.append((sequence.cpu().numpy(), prediction, error))
    
    return anomalies

# Example usage
threshold = 0.01  # This is an example threshold, adjust based on your error analysis
new_data = ...  # Your new data sequences, preprocessed similarly to the training data

anomalies = detect_anomalies(model, new_data, threshold)

print(f"Detected {len(anomalies)} anomalies.")


TypeError: 'ellipsis' object is not iterable